# LoRA Fine-Tuning for Information Extraction

## Overview

This notebook presents a **parameter-efficient fine-tuning approach using LoRA (Low-Rank Adaptation)** for structured information extraction from clinical trial abstracts in the **EBM-NLP dataset**. The objective is to extract key **PIO elements—Population, Intervention, and Outcome**—from unstructured biomedical text.

Unlike prompting-based methods such as zero-shot and few-shot learning, LoRA fine-tunes a pre-trained language model in a lightweight and computationally efficient manner by injecting trainable low-rank matrices into selected layers while keeping the majority of the model parameters frozen.

The pipeline includes:

- Preparing the EBM-NLP dataset in a structured input-output format suitable for supervised fine-tuning  
- Applying LoRA adapters to a pre-trained transformer model  
- Training the model to map clinical trial abstracts to structured PIO outputs  


In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

In [25]:
df = pd.read_excel(r"/content/drive/MyDrive/Colab Notebooks/cwk_data/cwk_train_data.xlsx")

In [5]:
df.columns.tolist()

['Unnamed: 0',
 'Doc_id',
 'Documents',
 'Tokens',
 'Intervention_Lables',
 'Outcome_Lables',
 'Participants_Label',
 'Participants_BIO_Labels',
 'Interventions_BIO_Lables',
 'Outcomes_BIO_Lables',
 'Merged BIO Labels']

In [16]:
import json
import ast

all_documents = []
df["Tokens"] = df["Tokens"].apply(ast.literal_eval)
df["Merged BIO Labels"] = df["Merged BIO Labels"].apply(ast.literal_eval)



for idx in range(300,450):
    document = df["Documents"][idx]
    tokens = df["Tokens"][idx]
    biolabels = df["Merged BIO Labels"][idx]

    extracted = {"Participants":[], "Intervention":[], "Outcome":[]}
    temp_text = []
    current = None

    for tok, lab in zip(tokens, biolabels):
        if lab.startswith("B"):
            if len(temp_text) > 0 and current:
                extracted[current].append(" ".join(temp_text))
                temp_text = []

            if "INT" in lab: current = "Intervention"
            elif "PART" in lab: current = "Participants"
            elif "OUT" in lab: current = "Outcome"
            temp_text.append(tok)

        elif lab.startswith("I"):
            temp_text.append(tok)

        elif lab.startswith("O"):
            if len(temp_text) > 0 and current:
                extracted[current].append(" ".join(temp_text))
                temp_text = []
                current = None

    if len(temp_text) > 0 and current:
        extracted[current].append(" ".join(temp_text))

    my_data_dict = {
        "instruction": "Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object.",
        "input": document,
        "output": json.dumps(extracted)
    }

    all_documents.append(my_data_dict)

with open(r"/content/drive/MyDrive/Colab Notebooks/Lora/training_set.jsonl", "w") as f:
    for entry in all_documents:
        f.write(json.dumps(entry) + "\n")


In [17]:
len(all_documents)

150

In [20]:
print(all_documents[1])

{'instruction': 'Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object.', 'input': 'Ipratropium bromide nasal spray for treatment of rhinorrhea in the laryngectomized patient: a pilot study. Many who have had a total laryngectomy complain of unrelenting rhinorrhea that is often very difficult to control. This study was undertaken to evaluate the effect of ipratropium bromide (IB), an anticholinergic nasal spray, on rhinorrhea in these patients. This was designed as a prospective, randomized, double-blind, placebo-controlled, crossover pilot study. Participants were selected if they had a total laryngectomy and complained of rhinorrhea. They were asked to rate the severity and duration of their rhinorrhea each day throughout the study on a scale from zero to six. Each participant was initially given a saline nasal spray for one week. They were then randomized to use either IB or saline for the double-blinded portion of the s

In [29]:
document = df["Documents"][2]
tokens = df["Tokens"][300]
m = df["Merged BIO Labels"][1]

In [30]:
document

'Pre-operative short-term pulmonary rehabilitation for patients of chronic obstructive pulmonary disease undergoing coronary artery bypass graft surgery. The role of pre-operative short-term pulmonary rehabilitation in patients with chronic obstructive pulmonary disease who undergo coronary artery bypass graft surgery has been assessed for the first time prospectively. Forty-five patients posted for coronary artery bypass graft surgery were randomised to receive either short-term pulmonary rehabilitation (group I) or no such programme (group II). Patients of both the groups were evenly matched with respect to age, sex, body surface area, duration and severity of chronic obstructive pulmonary disease and coronary artery disease. Normal individuals who evenly matched with the study group were assessed for normal respiratory function parameters. Pre-operative and post-operative peak expiratory flow rate, inspiratory capacity, post-operative ventilation time, post-operative pulmonary compl

In [15]:
print(tokens)

['Children', "'s", 'attitudes', 'and', 'behavioral', 'intentions', 'toward', 'a', 'peer', 'with', 'autistic', 'behaviors', ':', 'does', 'a', 'brief', 'educational', 'intervention', 'have', 'an', 'effect', '?', 'This', 'study', 'examined', 'children', "'s", 'ratings', 'of', 'attitudes', 'and', 'behavioral', 'intentions', 'toward', 'a', 'peer', 'presented', 'with', 'or', 'without', 'autistic', 'behaviors', '.', 'The', 'impact', 'of', 'information', 'about', 'autism', 'on', 'these', 'ratings', 'was', 'investigated', 'as', 'well', 'as', 'age', 'and', 'gender', 'effects', '.', 'Third-', 'and', 'sixth-grade', 'children', '(', 'N', '=', '233', ')', 'were', 'randomly', 'assigned', 'to', 'view', 'a', 'video', 'of', 'the', 'same', 'boy', 'in', 'one', 'of', 'three', 'conditions', ':', 'No', 'Autism', ',', 'Autism', ',', 'or', 'Autism/Information', '.', 'Children', 'at', 'both', 'grade', 'levels', 'showed', 'less', 'positive', 'attitudes', 'toward', 'the', 'child', 'in', 'the', 'two', 'autism', 'c

In [ ]:
# 1. Save the LoRA adapters locally in Colab
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. (Recommended) Copy the folder to your Google Drive so it's safe forever
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree("lora_model", "/content/drive/MyDrive/medical_lora_model")
